# Lekcija 05 - Agentični RAG


## Namestitev

Ta zvezek prikazuje vzorec Agentic RAG (generiranje z nadgradnjo iskanja) z uporabo Microsoftovega agentnega ogrodja.

**Predpogoji:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — vaš konec storitve Azure AI Search
- `AZURE_SEARCH_API_KEY` — vaš ključ API za Azure AI Search
- Implementacija Azure OpenAI nastavljena prek spremenljivk okolja
- Avtorizacija Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kaj je agentni RAG?

Tradicionalni RAG sledi fiksni poti: pridobi dokumente, nato ustvari odgovor. **Agentni RAG** gre korak dlje tako, da agentu daje samostojnost, da odloči **kdaj** in **kako** pridobiti informacije.

Z agentnim RAG lahko agent:
- **Odloči**, ali je za odgovor na vprašanje potreben postopek pridobivanja
- **Izbere**, kateri vir podatkov ali orodje bo poizvedoval
- **Oceni** pridobljene rezultate in izvede dodatne iskalne akcije, če prvi poskus ni zadosten
- **Združi** informacije iz več korakov pridobivanja v koherenten odgovor

To naredi agenta bolj prilagodljivega in natančnega v primerjavi s statično potjo pridobivanja- nato-generiranja.


## Ustvarjanje orodja za iskanje

V Agentic RAG so zunanji podatkovni viri oviti kot **orodja**, ki jih agent lahko po potrebi prikliče. To omogoča agentu, da obravnava pridobivanje informacij kot še eno dejanje, ki ga lahko izvede, namesto kot obvezen korak.

Spodaj definiramo bazo znanja o potovanjih in jo ponudimo kot orodje, ki ga agent lahko uporabi za iskanje informacij o destinacijah.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Ustvarjanje RAG agenta

Sedaj ustvarimo agenta, ki mu je naročeno, naj **vedno pridobi informacije pred odgovorom**. Agent uporablja orodje `search_travel_knowledge`, da utemelji svoje odgovore na bazi znanja, namesto da bi se zanašal na lastne podatke usposabljanja.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativno iskanje — vzorec tvorca-preverjevalca

Ključna prednost Agentic RAG je **iterativno iskanje**. Agent lahko opravlja več krogov iskanja, da preveri, izpopolni ali razširi svoje začetne ugotovitve — podobno kot delovni proces "tvorca-preverjevalca":

1. **Korak tvorca**: Agent pridobi začetne informacije in pripravi osnutek odgovora.  
2. **Korak preverjevalca**: Agent opravi dodatna iskanja, da preveri podrobnosti ali zapolni vrzeli.

Spodaj je agentu zastavljeno vprašanje, ki zahteva primerjavo več destinacij, kar ga spodbuja, da išče večkrat.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Povzetek

V tej lekciji ste se naučili, kako zgraditi **Agentic RAG** sistem z uporabo Microsoft Agent Framework:

- **Agentic RAG** agentom omogoča, da samostojno odločajo, kdaj pridobiti informacije, zaradi česar je pridobivanje dinamično in ne fiksno.
- **Orodja kot podatkovni viri**: Zunanje baze znanja (kot je Azure AI Search) so ovite kot orodja, ki jih agent lahko kliče.
- **Iterativno pridobivanje**: vzorec maker-checker omogoča agentu, da izvede več krogov pridobivanja — iskanje, preverjanje in izpopolnjevanje — preden poda končni odgovor.

V produkciji bi namesto lokalne `TRAVEL_KNOWLEDGE_BASE` uporabili pravi indeks Azure AI Search za upravljanje obsežnega iskanja po potovalnih dokumentih.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o omejitvi odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas opozarjamo, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem matičnem jeziku velja za avtoritativni vir. Za pomembne informacije priporočamo strokovni človeški prevod. Za kakršnekoli nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda, ne odgovarjamo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
